# The predictability screen

Pairwise correlation cannot answer "which columns are functions of the
rest": it misses interactions, mixed types, and anything nonlinear that
is not monotone. The predictability screen answers the question directly —
every eligible column is tried as a target against all the others, and a
small purpose-built gradient boosting model measures how predictable it
is on held-out rows. Its output is a `dependencies/predictive` statement:
named inputs, a target, a holdout score, and a majority baseline for
categorical targets.

## The encoded matrix

All model passes share one encoded matrix: a deterministic stratified
sample of at most 20,000 rows (`deep`: 50,000), every usable column
encoded compactly —

- numeric columns and date/datetime columns (as day numbers) into at most
  32 quantile bins, with the real bin edges retained so rules can quote
  actual thresholds;
- strings with at most 32 distinct levels into nominal codes (strings
  with more levels are skipped, and the skip is counted in coverage);
- booleans into two levels;
- at most 48 features (64 on `deep`) after this conversion.

The matrix is built once per profile; the predictability screen, the
decision-table pass, PRISM, and the association miner all read it. The
bin edges and level labels travel with the matrix, so every finding can
name conditions in original units.

## The model

The screener is an in-house histogram gradient boosting machine, built for
this job: depth-3 trees, shrinkage 0.3, L2 regularization λ = 1, at most
30 rounds with early stopping (patience 5) on a deterministic holdout.
Bins are packed row-major — LightGBM-style — so each round builds
histograms in a single pass over contiguous memory.

For each target: all present rows are collected (at least 300 required),
shuffled with a per-target seeded generator, and split 75/25 into train
and holdout; fitting uses at most 5,000 rows (10,000 on `deep`).
Predictors are shortlisted by normalized mutual information between the
target and every other feature over up to 2,048 training rows, truncated
to the feature cap; the target's own column never enters. Minimum leaf
size scales with the training rows (n/200, clamped to [10, 1,000]).

## What it reports

- Numeric targets: holdout R².
- Categorical targets (≤ 24 levels, or ≤ 12 distinct for a numeric
  target): holdout accuracy against the majority baseline.
- `inputs`: every predictor actually used by the fitted model — the
  complete list, not a top-three excerpt.
- Top features by gain: explanatory metadata, separate from the inputs,
  because gain ranking alone under-reports correlated contributors.

A finding is emitted when the holdout score clears the target-type
threshold. The evidence carries `n_train`, `n_holdout`, the score, the
baseline, `top_features`, and the seed-free but deterministic provenance.
Validation is a `diagnostic_holdout` that participated in early stopping —
it is marked `used_for_selection`, and it is not an independent
confirmation sample.

In [ ]:
import numpy as np
import polars as pl

import onceover

rng = np.random.default_rng(21)
n = 6_000
x1 = rng.uniform(-1, 1, n)
x2 = rng.uniform(-1, 1, n)
y = x1 * x2 + rng.normal(0, 0.05, n)      # interaction-only signal

df = pl.DataFrame({
    "x1": x1,
    "x2": x2,
    "y": y,
} | {f"noise_{i:02}": rng.random(n) for i in range(6)})

report = onceover.profile(df)
pred = [f for f in report.findings
        if f["category"] == "dependencies" and f.get("type") == "predictive"
        and f["subject"].get("target") == "y"]
print("predictive findings for y:", len(pred))
screen = [f for f in pred
          if any(e["method"]["detector"] == "gbm" for e in f["evidence"])]
for f in screen:
    e = next(e for e in f["evidence"] if e["method"]["detector"] == "gbm")
    print(" claim:", f["claim"])
    print(" inputs:", e["metrics"]["inputs"])
    print(" top features:", e["metrics"]["top_features"])

The interaction `x1 · x2` is invisible to pairwise correlation — each
feature alone is noise — yet the screen recovers it, because a boosted
tree splits on one feature and then the other. The screen can therefore detect interactions that pairwise
correlations miss.

Two detectors write to `dependencies/predictive` for this target: the
screen and the piecewise regime model. The loop selects the screen's
evidence by detector before printing gain, because "top features" is a
property of the fitted model rather than of the statement. Note that the
screen's input list includes all six noise columns: a depth-3 model with
30 rounds can touch many predictors, and the evidence reports what it
used rather than a curated top three.

## What the screen cannot do

A per-feature shortlist can drop a signal that only exists jointly (two
individually uninformative columns may both be shortlisted out on a wide
table). Early stopping on a small holdout can miss weak-but-real
structure. And a high R² says nothing about *how* the target is
determined — interpret the result as "these inputs predict the target at
this holdout score", and the exact-mapping question belongs to the
functional-dependency pass. The screen's caps (targets, features, rows)
are disclosed in `predictability_screen` coverage on every run.